In [1]:
import numpy as np
import math
import time
import os
import glob

import pandas as pd

from astropy.io.votable import parse
from astroquery.casda import Casda
from astroquery.utils.tap.core import TapPlus
from astroquery.utils.tap.core import Tap
from astropy.io.votable import parse, parse_single_table
from astropy.table import Table

import getpass

import astropy.coordinates as coord
from astropy.coordinates import SkyCoord
import astropy.units as un

casda = Casda()
casda.login(username=...)

TypeError: CasdaClass._login() takes 1 positional argument but 2 were given

In [2]:
def convert_xml_to_pandas(xml_file_name):
    votable = parse(xml_file_name)
    table = votable.get_first_table()
    bill = table.to_table(use_names_over_ids=True)
    return bill.to_pandas()

In [3]:
# Set up the TAP url
tap = TapPlus(url="https://casda.csiro.au/casda_vo_tools/tap")

In [4]:
# This is the search function, here I'm searching onl
# for continuum catalogues. You can change this to search
# for other data products instead
job = tap.launch_job_async(("SELECT TOP 50000 * FROM ivoa.obscore where(dataproduct_subtype = 'catalogue.continuum.component')"))
r = job.get_results()

INFO: Query finished. [astroquery.utils.tap.core]


In [5]:
# Here I keep only good or uncertain data
data = r[(r['quality_level'] == 'GOOD') | (r['quality_level'] == 'UNCERTAIN')]

In [6]:
# You have to do this step unless you have permission
# for embargoed data associated with you OPAL
# account login
public_data = Casda.filter_out_unreleased(data)

In [7]:
# Get the centre coords of all of the observations
# in the table
public_coords = SkyCoord(np.array(public_data['s_ra'])*un.deg,
                         np.array(public_data['s_dec'])*un.deg)

In [8]:
# Get the file names
public_files = np.array(public_data['filename'])

In [9]:
# This is the filepath where this script
# will save the products you download
casda_filepath = '/import/ada1/ldri3523/SmallConvenienceNotebooks/'

In [10]:
# I like pandas better
pubdat = public_data.to_pandas()

print('Number of rows: ', len(pubdat.index))

Number of rows:  18061


In [11]:
# The coordinates of the source you're interested in
example_source_coords = SkyCoord(256.235 * un.deg, -48.350 * un.deg)
# This sets how far away from the centre of the image
# that you're searching
example_radius = 3.0 * un.deg

In [12]:
# Find the rows in the public data table
# that are within example_radius of your
# source
seps = example_source_coords.separation(public_coords)
matches = np.where(seps < example_radius)[0]

In [13]:
# Do a little coordinate check to
# Make sure you're getting the files you want
pubdat.iloc[matches][['s_ra', 's_dec']]

,s_ra,s_dec
1297,258.826794,-46.586269
3287,258.748789,-46.506017
4850,257.748852,-51.134900
4884,258.772706,-46.516292
8609,258.748789,-46.506017
10149,258.748487,-46.506781
10654,257.725058,-51.123304
12875,257.725501,-51.122332
13931,258.838380,-46.502927
17042,258.476307,-50.120760


In [14]:
matching_files = np.array(pubdat.iloc[matches]['filename'])
url_list = []

# This part stages the files you want to download
# so it sometimes takes a minute
for mfile in matching_files:
    pdata = public_data[public_data['filename'] == mfile]
    url = casda.stage_data(pdata)
    if url not in url_list:
        url_list += url

In [15]:
# Now download your files :D
filelist = casda.download_files(url_list,
                                savedir=casda_filepath)

INFO: Found cached file /import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.RACS_1714-46.SB20313.cont.taylor.0.restored.conv.components.xml.checksum with size 32 = 32. [astroquery.query]
INFO: Found cached file /import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.RACS_1714-46.SB20313.cont.taylor.0.restored.conv.components.xml with size 3050522 = 3050522. [astroquery.query]
INFO: Found cached file /import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.EMU_1714-46.SB32526.cont.taylor.0.restored.conv.v2.components.xml.checksum with size 32 = 32. [astroquery.query]
INFO: Found cached file /import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.EMU_1714-46.SB32526.cont.taylor.0.restored.conv.v2.components.xml with size 19450969 = 19450969. [astroquery.query]
INFO: Found cached file /import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.RACS_1710-51.SB36609.cont.taylor.0.restored.conv.components.xml.checksum with size 32 = 32. [astroquery.query]
IN

In [16]:
filelist

['/import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.RACS_1714-46.SB20313.cont.taylor.0.restored.conv.components.xml.checksum',
 '/import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.RACS_1714-46.SB20313.cont.taylor.0.restored.conv.components.xml',
 '/import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.EMU_1714-46.SB32526.cont.taylor.0.restored.conv.v2.components.xml.checksum',
 '/import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.EMU_1714-46.SB32526.cont.taylor.0.restored.conv.v2.components.xml',
 '/import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.RACS_1710-51.SB36609.cont.taylor.0.restored.conv.components.xml.checksum',
 '/import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.RACS_1710-51.SB36609.cont.taylor.0.restored.conv.components.xml',
 '/import/ada1/ldri3523/SmallConvenienceNotebooks/selavy-image.i.RACS_1714-46.SB36661.cont.taylor.0.restored.conv.components.xml.checksum',
 '/import/ada1/ldri3523/SmallConvenienceNot

In [17]:
pwd

'/import/ada1/ldri3523/SmallConvenienceNotebooks'

# Downloading cutouts

In [18]:
# Use these coordinates as an example. Let's say I want cutouts centred on
# these coordinates
source_coords = SkyCoord('17h04m56.2s -48d21m00.0s')
source_name = 'MKTJ170456.2-482100'

# Directory where I'm going to save the cutouts I make
cutout_path = '/import/ada1/ldri3523/SmallConvenienceNotebooks/CASDA_TAP_path/'

In [19]:
# Set up the TAP+ query for Stokes I cutouts
image_type = 'I'
tap_qry = (F"SELECT * FROM ivoa.obscore where(pol_states = '/{image_type}/' and "
           F"dataproduct_subtype = 'cont.restored.t0') "
           F"AND 1 = CONTAINS(POINT('ICRS',{source_coords.ra.deg},{source_coords.dec.deg}),s_region)")
# To download noise files: 'cont.noise.t0'

In [20]:
# Launch the query
job = tap.launch_job_async(tap_qry)
# Get the results table
r = job.get_results()
# Only include released data
r = Casda.filter_out_unreleased(r)
# Turn it into a pandas table (I prefer them)
r_df = r.to_pandas()

# Sometimes other telescopes are in the table,
# stick to only ASKAP
r_df = r_df[r_df['obs_id'].str.contains('ASKAP')]
# Remove data labelled "BAD"
r_df = r_df[r_df['quality_level'] != 'BAD']
# Remove other file types that I'm not interested in
r_df = r_df[~r_df['filename'].str.contains('raw')]
r_df = r_df[~r_df['filename'].str.contains('alt')]
r_df = r_df[~r_df['filename'].str.contains('highres')]
r_df = r_df[~r_df['filename'].str.contains('iqr')]
r_df = r_df[~r_df['obs_collection'].str.contains('BETA')]

INFO: Query finished. [astroquery.utils.tap.core]


In [21]:
# See what the table looks like
r_df

,dataproduct_type,calib_level,obs_collection,obs_id,obs_publisher_did,access_url,access_format,access_estsize,target_name,s_ra,...,em_unit,em_resolution,s_resolution_min,s_resolution_max,s_ucd,s_unit,obs_release_date,quality_level,thumbnail_id,filename
0,cube,2,ASKAP Pilot Survey for GASKAP-HI,ASKAP-13531,cube-29061,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,1669005,,252.191226,...,m,1.487820e-10,6.218482,7.660430,pos.eq,deg,2021-06-21T05:16:13.162Z,UNCERTAIN,<NA>,image.i.SB13531.cont.restored.fits
1,cube,2,ASKAP Pilot Survey for GASKAP-HI,ASKAP-13536,cube-28770,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,1669005,,252.191226,...,m,1.487820e-10,6.184402,7.727108,pos.eq,deg,2021-06-21T05:17:24.076Z,UNCERTAIN,<NA>,image.i.SB13536.cont.restored.fits
2,cube,2,The Rapid ASKAP Continuum Survey,ASKAP-13601,cube-26213,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,775840,,249.245516,...,m,3.806210e-10,11.376968,14.262429,pos.eq,deg,2020-12-17T02:38:18.222Z,GOOD,<NA>,image.i.SB13601.cont.RACS_1636-50A.linmos.tayl...
3,cube,2,The Rapid ASKAP Continuum Survey,ASKAP-13601,cube-26214,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,775840,,249.245516,...,m,3.806210e-10,13.200000,15.700000,pos.eq,deg,2020-12-17T02:38:18.222Z,GOOD,<NA>,image.i.SB13601.cont.RACS_1636-50A.linmos.tayl...
4,cube,2,The Rapid ASKAP Continuum Survey,ASKAP-20312,cube-75681,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,549436,,257.810616,...,m,1.603143e-10,8.000000,9.200000,pos.eq,deg,2023-04-26T07:29:46.560Z,GOOD,<NA>,image.i.RACS_1710-51.SB20312.cont.taylor.0.res...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,cube,2,The Rapid ASKAP Continuum Survey,ASKAP-8593,cube-19154,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,767166,,251.172787,...,m,3.806210e-10,12.100000,22.300000,pos.eq,deg,2020-11-16T03:49:29.551Z,GOOD,<NA>,image.i.SB8593.cont.RACS_1644-43A.linmos.taylo...
159,cube,2,The Rapid ASKAP Continuum Survey,ASKAP-8593,cube-19162,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,767166,,259.544887,...,m,3.806210e-10,10.257697,20.304001,pos.eq,deg,2020-11-16T03:49:29.551Z,GOOD,<NA>,image.i.SB8593.cont.RACS_1718-43A.linmos.taylo...
160,cube,2,The Rapid ASKAP Continuum Survey,ASKAP-8593,cube-19163,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,767166,,259.544887,...,m,3.806210e-10,12.100000,25.000000,pos.eq,deg,2020-11-16T03:49:29.551Z,GOOD,<NA>,image.i.SB8593.cont.RACS_1718-43A.linmos.taylo...
161,cube,2,The Rapid ASKAP Continuum Survey,ASKAP-8598,cube-19270,https://data.csiro.au/casda_vo_proxy/vo/datali...,application/x-votable+xml;content=datalink,775840,,258.476307,...,m,3.806210e-10,10.590075,18.500001,pos.eq,deg,2020-11-16T03:49:36.501Z,GOOD,<NA>,image.i.SB8598.cont.RACS_1713-50A.linmos.taylo...


In [22]:
# You really can't download 100+ cutouts all at once. So I split
# this into smaller batches. Here I'm doing batches of 5
url_loop_length = 5

# The size of the cutout. This will be the length
# of each edge of the square cutout
cutout_width = 4 * un.arcmin

print('Staging the noise files')
clean_url_list = []
r_filenames = np.array(r_df['filename'])
# I'm keeping it to one loop here, for demonstration purposes
for url_ in np.arange((len(r_filenames) // url_loop_length) + 1)[:2]:
    url_filenames = r_filenames[url_ * url_loop_length: (url_ + 1) * url_loop_length]
    print(url_filenames)
    url_r_df = r_df[r_df['filename'].isin(url_filenames)]
    url_list = casda.cutout(Table.from_pandas(url_r_df),
                            coordinates=source_coords, radius=cutout_width)
    # I find it annoying to download the checksum files
    # with the files I actually want
    for row in url_list:
        if 'checksum' not in row:
            clean_url_list.append(row)
            
    print('Downloading the files')
    # Now we can download that batch of files
    filelist = casda.download_files(clean_url_list,
                                    savedir=cutout_path)
    
    # Here I rename the downloaded files. They come out with
    # quite nasty filenames that don't give you wany real info.
    # So I give them back their original names
    for cf, cutf in enumerate(filelist):
        cube_id = 'cube-' + (cutf.split('-imagecube-')[-1]).split('.fits')[0]
    
        source_row = r_df[r_df['obs_publisher_did'] == cube_id]
    
        og_name = source_row['filename'].item()
    
        new_name = F'{cutout_path}{source_name}_{og_name}'
    
        os.rename(cutf, new_name)
    # Changing the file names does mean that you'll have to do
    # checks yourself to avoid downloading the same cutout twice
    clean_url_list = []

Staging the noise files
['image.i.SB13531.cont.restored.fits' 'image.i.SB13536.cont.restored.fits'
 'image.i.SB13601.cont.RACS_1636-50A.linmos.taylor.0.restored.fcor.fits'
 'image.i.SB13601.cont.RACS_1636-50A.linmos.taylor.0.restored.cres.fcor.fits'
 'image.i.RACS_1710-51.SB20312.cont.taylor.0.restored.conv.fits']


['image.i.RACS_1714-46.SB20313.cont.taylor.0.restored.conv.fits'
 'image.i.RACS_1644-46.SB20439.cont.taylor.0.restored.conv.fits'
 'image.i.EMU_1644-46.SB32043.cont.taylor.0.restored.fits'
 'image.i.EMU_1644-46.SB32043.cont.taylor.0.restored.conv.v2.fits'
 'image.i.EMU_1714-46.SB32526.cont.taylor.0.restored.fits']


Your cutouts should now be in the folder, ready to look at